# 07 – Working with Dates in Pandas

Dates are everywhere in real data — order dates, transaction dates, birth dates, login timestamps.

The problem is that when you load a CSV file, pandas doesn't automatically know that `"2024-01-15"` is a date. It reads it as plain text — just a string. You can't do any date math on a string.

Once you convert it to a proper **datetime** type, pandas unlocks a whole set of tools:
- Extract the year, month, day, weekday
- Filter rows by date range
- Calculate the difference between two dates
- Group data by month or quarter

This notebook covers all of that.

## 1) Converting Strings to Dates with `pd.to_datetime()`

This is always the first step — convert your date column from a string to a proper datetime type.

In [ ]:
import pandas as pd

# A simple string date — pandas reads this as object (text) by default
date_string = "2024-06-15"
print(type(date_string))        # plain Python string

# Convert it to a datetime
date = pd.to_datetime(date_string)
print(date)
print(type(date))               # now it's a Timestamp

In [ ]:
# pandas understands many common date formats automatically

print(pd.to_datetime("15-06-2024"))
print(pd.to_datetime("June 15, 2024"))
print(pd.to_datetime("15/06/2024"))
print(pd.to_datetime("2024/06/15"))

pandas is smart enough to recognise most standard formats without you specifying anything. But if your dates are in an unusual format, you can tell pandas exactly how to read it using the `format` parameter.

In [ ]:
# Specifying format manually — useful for ambiguous or unusual formats
# %d = day, %m = month, %Y = 4-digit year

print(pd.to_datetime("15-06-2024", format="%d-%m-%Y"))
print(pd.to_datetime("06/15/2024", format="%m/%d/%Y"))   # US format

## 2) Working with a Date Column in a DataFrame

In practice, you'll be converting an entire column — not just a single value.

In [ ]:
orders = pd.DataFrame({
    "Order_ID":    [1, 2, 3, 4, 5, 6],
    "Customer":    ["Amit", "Priya", "Ravi", "Neha", "Karan", "Sneha"],
    "Order_Date":  ["2024-01-10", "2024-02-15", "2024-02-28",
                    "2024-03-05", "2024-03-20", "2024-04-01"],
    "Amount":      [1500, 3200, 800, 4500, 2100, 950]
})

print(orders.dtypes)    # Order_Date is currently 'object' (string)

In [ ]:
# Convert the column to datetime

orders["Order_Date"] = pd.to_datetime(orders["Order_Date"])

print(orders.dtypes)    # Now it's datetime64

In [ ]:
print(orders)

The dates look the same in the output, but the dtype is now `datetime64[ns]` — and that unlocks everything below.

## 3) Extracting Parts of a Date with `.dt`

Once a column is datetime, the `.dt` accessor gives you access to individual components — year, month, day, weekday, and more.

In [ ]:
# Extract year, month, day separately

orders["Year"]  = orders["Order_Date"].dt.year
orders["Month"] = orders["Order_Date"].dt.month
orders["Day"]   = orders["Order_Date"].dt.day

print(orders)

In [ ]:
# Day of the week — Monday=0, Sunday=6

orders["Weekday_Num"]  = orders["Order_Date"].dt.dayofweek
orders["Weekday_Name"] = orders["Order_Date"].dt.day_name()

print(orders[["Order_Date", "Weekday_Num", "Weekday_Name"]])

In [ ]:
# Month name instead of month number

orders["Month_Name"] = orders["Order_Date"].dt.month_name()

print(orders[["Order_Date", "Month", "Month_Name"]])

In [ ]:
# Quarter of the year (Q1, Q2, Q3, Q4)

orders["Quarter"] = orders["Order_Date"].dt.quarter

print(orders[["Order_Date", "Quarter"]])

These extracted columns are very useful for grouping. For example, once you have a `Month` column, you can `groupby("Month")` to get monthly totals.

## 4) Filtering by Date

Once a column is datetime, you can filter rows using date comparisons — just like filtering numbers.

In [ ]:
# Orders placed after February 1st 2024

after_feb = orders[orders["Order_Date"] > "2024-02-01"]
print(after_feb[["Order_ID", "Customer", "Order_Date", "Amount"]])

In [ ]:
# Orders within a specific date range — February only

feb_orders = orders[
    (orders["Order_Date"] >= "2024-02-01") &
    (orders["Order_Date"] <= "2024-02-28")
]
print(feb_orders[["Order_ID", "Customer", "Order_Date", "Amount"]])

In [ ]:
# Filter by month number — all March orders

march_orders = orders[orders["Order_Date"].dt.month == 3]
print(march_orders[["Order_ID", "Customer", "Order_Date", "Amount"]])

## 5) Calculating the Difference Between Two Dates

When you subtract one datetime column from another, you get a **timedelta** — a duration.

In [ ]:
# Add a delivery date column

orders["Delivery_Date"] = pd.to_datetime([
    "2024-01-14", "2024-02-19", "2024-03-03",
    "2024-03-08", "2024-03-25", "2024-04-05"
])

# Calculate how many days it took for delivery
orders["Days_to_Deliver"] = orders["Delivery_Date"] - orders["Order_Date"]

print(orders[["Order_Date", "Delivery_Date", "Days_to_Deliver"]])

The result shows `4 days`, `3 days`, etc. — but the dtype is `timedelta64`, not a plain integer.

To get just the number of days as an integer, use `.dt.days`.

In [ ]:
# Extract just the number as an integer

orders["Days_to_Deliver"] = (orders["Delivery_Date"] - orders["Order_Date"]).dt.days

print(orders[["Customer", "Days_to_Deliver"]])

**Why `.normalize()`?**

`pd.Timestamp("today")` gives you today's date AND the current time — e.g. `2024-06-15 14:32:11.458`.

If you subtract that from a date like `2024-01-10` (which has no time component and defaults to midnight), you get a fractional number of days — `156.60 days` instead of a clean `156 days`.

`.normalize()` strips the time part and sets it to midnight — so you're comparing date to date cleanly.

In [ ]:
# How many days since today did each order happen?

today = pd.Timestamp("today").normalize()   # today's date with time set to midnight
orders["Days_Ago"] = (today - orders["Order_Date"]).dt.days

print(orders[["Customer", "Order_Date", "Days_Ago"]])

## 6) Grouping by Date Parts

Once you've extracted date components into columns, you can group on them just like any other column.

In [ ]:
# Total order amount per month

monthly = orders.groupby("Month_Name")["Amount"].sum().reset_index()
monthly.columns = ["Month", "Total_Amount"]
print(monthly)

In [ ]:
# Total amount per quarter

quarterly = orders.groupby("Quarter")["Amount"].sum().reset_index()
quarterly.columns = ["Quarter", "Total_Amount"]
print(quarterly)

In [ ]:
# How many orders on each day of the week?

print(orders.groupby("Weekday_Name")["Order_ID"].count())

## 7) Sorting by Date

Datetime columns sort correctly with `sort_values()` — no special handling needed.

In [ ]:
# Sort orders from oldest to newest

print(
    orders[["Customer", "Order_Date", "Amount"]]
    .sort_values("Order_Date")
    .reset_index(drop=True)
)

## 8) Creating a Date Range with `pd.date_range()`

Sometimes you need to generate a sequence of dates — for example, to fill in missing days or build a calendar.

In [ ]:
# Generate daily dates for one week

dates = pd.date_range(start="2024-01-01", end="2024-01-07", freq="D")
print(dates)

In [ ]:
# Generate the first day of each month for a full year

monthly_dates = pd.date_range(start="2024-01-01", periods=12, freq="MS")  # MS = Month Start
print(monthly_dates)

In [ ]:
# Build a simple calendar DataFrame

calendar = pd.DataFrame({"Date": pd.date_range("2024-01-01", periods=30, freq="D")})
calendar["Day"]       = calendar["Date"].dt.day
calendar["Weekday"]   = calendar["Date"].dt.day_name()
calendar["Is_Weekend"] = calendar["Date"].dt.dayofweek >= 5   # 5=Sat, 6=Sun

print(calendar.head(10))

## Putting It All Together

Let's do a realistic end-to-end example — analyse a sales dataset by date.

In [ ]:
# Raw sales data
sales = pd.DataFrame({
    "Sale_Date":  ["2024-01-05", "2024-01-20", "2024-02-10",
                   "2024-02-25", "2024-03-08", "2024-03-22",
                   "2024-04-03", "2024-04-18"],
    "Product":   ["Laptop", "Phone", "Laptop", "Tablet",
                  "Phone", "Laptop", "Tablet", "Phone"],
    "Amount":    [55000, 18000, 62000, 25000, 21000, 58000, 27000, 19000]
})

# Step 1 — convert to datetime
sales["Sale_Date"] = pd.to_datetime(sales["Sale_Date"])

# Step 2 — extract useful date parts
sales["Month"]   = sales["Sale_Date"].dt.month
sales["Quarter"] = sales["Sale_Date"].dt.quarter

# Step 3 — monthly revenue summary
monthly_revenue = (
    sales.groupby("Month")["Amount"]
    .sum()
    .reset_index()
    .rename(columns={"Amount": "Revenue"})
)
print("Monthly Revenue:")
print(monthly_revenue)
print()

# Step 4 — best selling product per quarter
product_by_quarter = (
    sales.groupby(["Quarter", "Product"])["Amount"]
    .sum()
    .reset_index()
    .sort_values(["Quarter", "Amount"], ascending=[True, False])
)
print("Sales by Product per Quarter:")
print(product_by_quarter)

## Summary

- Always convert date columns with `pd.to_datetime(df["col"])`
- Use `format=` if the date string is ambiguous or unusual
- After converting, use `.dt` to extract parts:
  - `.dt.year`, `.dt.month`, `.dt.day`
  - `.dt.day_name()`, `.dt.month_name()`
  - `.dt.dayofweek`, `.dt.quarter`
- Filter by date using `>`, `<`, `>=`, `<=` with a date string
- Subtract two datetime columns to get a timedelta — use `.dt.days` for just the number
- Group by extracted date parts (month, quarter, weekday) for time-based summaries
- Generate date sequences with `pd.date_range()`

Next up: **String Operations in Pandas** — cleaning and transforming text columns.

## Practice Questions 1

1. Create a DataFrame with 6 employee records — columns: `Name`, `Join_Date`, `Salary`. Use string dates.
2. Convert `Join_Date` to datetime.
3. Extract the year and month into separate columns.
4. Filter employees who joined after a date of your choice.

## Practice Questions 2

**Note:** `NaT` stands for **Not a Time** — it is the datetime equivalent of `NaN`. Use it when a datetime value is missing.

```python
import pandas as pd
import numpy as np
pd.NaT   # missing datetime value
```

1. Add an `Exit_Date` column to a small employee DataFrame — use `pd.NaT` for employees still working.
2. Calculate `Tenure_Days` — number of days each employee worked (use today's date for active employees).
3. Group employees by the year they joined and find the average salary per year.